# Structured Outputs & Constrained Decoding Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: regex-constrained generation from scratch

See `code/main.py` for a standalone FSM implementation. The core idea in 30 lines:

In [ ]:
```python

def mask_logits(logits, valid_token_ids):

    mask = [float("-inf")] * len(logits)

    for tid in valid_token_ids:

        mask[tid] = logits[tid]

    return mask

def generate_constrained(model, tokenizer, prompt, fsm):

    ids = tokenizer.encode(prompt)

    state = fsm.initial_state

    while not fsm.is_accept(state):

        logits = model.next_token_logits(ids)

        valid = fsm.valid_tokens(state, tokenizer)

        logits = mask_logits(logits, valid)

        tok = sample(logits)

        ids.append(tok)

        state = fsm.transition(state, tok)

    return tokenizer.decode(ids)

In [ ]:
```

The FSM tracks what parts of the grammar we have satisfied so far. `valid_tokens(state, tokenizer)` computes which vocabulary tokens can advance the FSM without leaving an accepting path.

### Step 2: Outlines for JSON Schema

In [ ]:
```python

from pydantic import BaseModel

from typing import Literal

import outlines

class Review(BaseModel):

    sentiment: Literal["positive", "negative", "neutral"]

    confidence: float

    evidence_span: str

model = outlines.models.transformers("meta-llama/Llama-3.2-3B-Instruct")

generator = outlines.generate.json(model, Review)

result = generator("Classify: 'The wait staff was attentive and the food arrived hot.'")

print(result)

# Review(sentiment='positive', confidence=0.93, evidence_span='attentive ... hot')

In [ ]:
```

Zero validation errors. Ever. The FSM makes invalid output unreachable.

### Step 3: Instructor for provider-agnostic Pydantic

In [ ]:
```python

import instructor

from anthropic import Anthropic

from pydantic import BaseModel, Field

class Invoice(BaseModel):

    vendor: str

    total_usd: float = Field(ge=0)

    line_items: list[str]

client = instructor.from_anthropic(Anthropic())

invoice = client.messages.create(

    model="claude-opus-4-7",

    max_tokens=1024,

    response_model=Invoice,

    messages=[{"role": "user", "content": "Extract from: 'Acme Corp $420. Widget, Gizmo.'"}],

)

In [ ]:
```

Different mechanism. Instructor does not touch logits. It formats the schema into the prompt, parses the output, and retries on validation failure (default 3 times). Works with any provider. Retries add latency and cost. Cross-provider portability is the selling point.

### Step 4: native vendor APIs

In [ ]:
```python

from openai import OpenAI

client = OpenAI()

response = client.responses.create(

    model="gpt-5",

    input=[{"role": "user", "content": "Classify: 'The food was cold.'"}],

    text={"format": {"type": "json_schema", "name": "sentiment",

          "schema": {"type": "object", "required": ["sentiment"],

                     "properties": {"sentiment": {"type": "string",

                                                  "enum": ["positive", "negative", "neutral"]}}}}},

)

print(response.output_parsed)

In [ ]:
```

Server-side constrained decoding. Reliability parity with Outlines for supported schemas. No local model management. Locks you to the vendor.

## Exercises

In [ ]:
1. **Easy.** Prompt a small open-weights model (e.g., Llama-3.2-3B) without constrained decoding for `Review(sentiment, confidence, evidence_span)`. Measure the fraction that parse as valid JSON on 100 reviews.
2. **Medium.** Same corpus with Outlines JSON mode. Compare compliance rate, latency, and semantic accuracy.
3. **Hard.** Implement a regex-constrained decoder from scratch for phone numbers (`\d{3}-\d{3}-\d{4}`). Verify 0 invalid outputs on 1000 samples.